# Experimental Results Leaderboard

This page provides an interactive view of results for 16 models across 4 datasets, aggregated from over 1900 individual runs.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.renderers.default = 'notebook_connected'

# Load Summarized and Raw data
base_path = Path('_static') if Path('_static').exists() else Path('.') / '_static'
df = pd.read_csv(base_path / 'leaderboard_data.csv')
df_raw = pd.read_csv(base_path / 'leaderboard_raw.csv')

def get_color_map(methods):
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Bold
    unique_methods = sorted(list(set(methods)))
    return {m: colors[i % len(colors)] for i, m in enumerate(unique_methods)}

color_map = get_color_map(df['Method'].unique())

## 🏆 Top 3 Methods Comparison (Radar Chart)
This chart compares the top 3 best-performing models for each dataset across multiple metrics.

In [ ]:
for ds in df['Dataset'].unique():
    ds_df = df[df['Dataset'] == ds].sort_values('Test', ascending=False).head(3)
    fig = go.Figure()
    for _, row in ds_df.iterrows():
        fig.add_trace(go.Scatterpolar(
            r=[row['Train'], row['Test'], row['Test'] - row['Test Std'], row['Test'] + row['Test Std']],
            theta=['Train Acc', 'Test Acc', 'Lower Bound', 'Upper Bound'],
            fill='toself', name=row['Method'],
            line=dict(color=color_map.get(row['Method']))
        ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])), 
        title=f"Top 3 Profile: {ds.upper()}", template='plotly_white'
    )
    fig.show()

## 📊 Statistical Summary Table

In [ ]:
pdf = df.sort_values(['Dataset', 'Test'], ascending=[True, False])
fig = go.Figure(data=[go.Table(
    header=dict(values=[f"<b>{c}</b>" for c in pdf.columns], fill_color='paleturquoise', align='left'),
    cells=dict(values=[pdf[c] for c in pdf.columns], format=[None, None, '.4f', '.4f', None, None, '.4f', '.4f', None, None],
               fill_color='lavender', align='left'))
])
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=600)
fig.show()

## 📈 Global Performance Heatmap
A bird's-eye view of how all models perform across all datasets.

In [ ]:
pivot_df = df.pivot(index='Method', columns='Dataset', values='Test')
fig = px.imshow(pivot_df, text_auto=".3f", aspect="auto", color_continuous_scale='Viridis',
                title="Model Generalization across Datasets", template='plotly_white')
fig.show()

## 🛡️ Stability Analysis (Box Plots)
Visualizing the variance across all 30 runs for each model.

In [ ]:
for ds in df_raw['Dataset'].unique():
    fig = px.box(df_raw[df_raw['Dataset'] == ds], x='Method', y='Test Accuracy', color='Method', 
                 color_discrete_map=color_map, points='all', template='plotly_white',
                 title=f"Stability Distribution: {ds.upper()}")
    fig.show()

## 🎯 Efficiency Frontier
Comparison of Training vs Testing accuracy. Ideally, models should be in the top-right corner with a small gap.

In [ ]:
fig = px.scatter(df, x="Test", y="Train", size="Test Std", color="Method", 
                 facet_col="Dataset", hover_name="Method", color_discrete_map=color_map, 
                 template="plotly_white", title="Training vs Testing Performance")
fig.update_layout(height=400)
fig.show()